# GenAI Pipeline — End Product Testing Notebook

LLM-based end product / sub-end product labelling of grants on alternative proteins.
Uses Claude with prompt caching via the Anthropic Python SDK. Adapted from
`grant_rescat_testing.ipynb` — see `4_endproduct/v2/NOTE.txt` for what changed in the prompt.


### 1. Imports and Configuration


In [1]:
import pandas as pd
import json
import random
import time
import anthropic
import os
from pathlib import Path
from dotenv import load_dotenv
from pydantic import create_model

load_dotenv()

OUTPUT_DIR = Path(".")
RAW_SUBS_DIR = Path("4_endproduct/raw_subsets/")


### 2. Data Inspection


In [2]:
EXCEL_PATH = Path("1_deduplication/raw_data/Funding2026_inscope.xlsx")
df_raw = pd.read_excel(EXCEL_PATH)

# Only ~38% of records have an End product type label at all, and this pipeline stage
# isn't split by AP pillar, so every in-scope record is eligible regardless of production
# platform — just require an End product type label (to sample/score against) and a
# non-empty Abstract (so the LLM has enough text to classify). Records with a label but no
# abstract are set aside for manual review instead of silently dropped.
has_endproduct = df_raw["End product type"].notna() & (df_raw["End product type"].str.strip() != "")
has_abstract = df_raw["Abstract"].notna() & (df_raw["Abstract"].str.strip() != "")

df = df_raw[has_endproduct & has_abstract].reset_index(drop=True)
needs_manual_review = df_raw[has_endproduct & ~has_abstract].reset_index(drop=True)

# "sub-end product" has a stray trailing-space variant of "Milk" in the raw data
# ('Milk ' vs 'Milk') — strip whitespace so it isn't treated as a distinct label.
df["sub-end product"] = df["sub-end product"].str.strip()

print(f"Raw shape: {df_raw.shape}")
print(f"Filtered shape (End product type not empty, has abstract): {df.shape}")
print(f"Set aside for manual review (label present, no abstract): {needs_manual_review.shape[0]}")
print(f"\nColumns: {list(df.columns)}")
df.head()


Raw shape: (1678, 81)
Filtered shape (End product type not empty, has abstract): (340, 81)
Set aside for manual review (label present, no abstract): 302

Columns: ['Title', 'Abstract', 'Original title', 'Database', 'Total amount', 'Gov contribution', 'Currency', 'Total amount (USD)', 'Gov contribution (USD)', 'Total amount (EUR)', 'Gov & NP contribution (EUR)', 'Funding decision', 'copy to external database', 'URL for announcement', 'Identification code', 'Unnamed: 15', 'Unnamed: 16', 'Notes (external)', 'Notes (internal)', 'Project lead (PI)', 'PI department', 'PI organisation', 'PI organisation type', 'PI organisation country', 'PI organisation region', 'PI organisation state', 'PI organisation zip code', 'PI organisation congressional district', 'Collaborator names', 'Collaborator institutions', 'Multiple organisation recipients', 'Date request submitted', 'Year request submitted', 'Date award announced', 'Project start date', 'Duration of award (months)', 'Project status', 'Annual 

,Title,Abstract,Original title,Database,Total amount,Gov contribution,Currency,Total amount (USD),Gov contribution (USD),Total amount (EUR),...,2026,2027,2028,2029,2030,2031,2032,2033,2034,2035
0,National Alternative Proteins Innovation and K...,"To secure a continued supply of safe, tasty, ...",NaN,airtable,38000000,16001352,GBP,48859450.0,18697500.0,45220000.0,...,3173601.48,3173601.48,3173601.48,3173601.48,NaN,NaN,NaN,NaN,NaN,NaN
1,Polish Government Backs LabFarm With Grant for...,The grant will enable LabFarm to optimize biop...,NaN,airtable,9000000,9000000,PLN,2324250.0,2324250.0,2070000.0,...,690000.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,"New sustainable proteins for food, feed and no...",InnoProtein addresses this challenge by tappin...,NaN,airtable,5043848,4592392,EUR,5447355.0,4959783.0,5043848.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Marie Skłodowska-Curie Action Edible Soft Matter,Food and environmental transitions are worldw...,NaN,airtable,4041810,4041810,EUR,4041810.0,4041810.0,4041810.0,...,808362.00,808362.00,808362.00,808362.00,NaN,NaN,NaN,NaN,NaN,NaN
4,CERAFIM - Cellular Agriculture for Sustainable...,"Cellular agriculture produces proteins, fats a...",NaN,airtable,1266000,1266000,EUR,1356136.0,1356136.0,1266000.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### 3. Balanced Subset Creation

Take up to `N_PER_CATEGORY` records per End product type category. A multi-label row (e.g.
"Meat, Dairy") is eligible under each of its labels and is kept only once in the final set if
selected more than once.

Unlike `genai_rescat_testing.ipynb`, this stage isn't split by AP pillar / production platform —
every in-scope, labelled record is eligible regardless of platform.

Also top up the sample so at least `MIN_PER_COMBO` examples of each distinct multi-label
combination are included — this specifically tests the LLM's multi-label (true/false per
category) classification.


In [3]:
# End product type can hold multiple comma-separated labels per row; explode before counting.
def explode_categories(d):
    return d["End product type"].str.split(",").explode().str.strip()

all_categories = sorted(explode_categories(df).dropna().unique())

breakdown = explode_categories(df).value_counts().reindex(all_categories, fill_value=0)
breakdown.index.name = "end_product_type"
breakdown = breakdown.to_frame(name="count")
breakdown


,count
end_product_type,
Agnostic,220
Dairy,35
Eggs,4
Fats,6
Meat,69
Seafood,11
Snacks,1
Sugar,1


In [4]:
RANDOM_STATE = 3
N_PER_CATEGORY = 20
MIN_PER_COMBO = 2  # ensure at least this many examples of each distinct multi-label combination

def parse_categories(series):
    """Split a comma-separated 'End product type' string into a cleaned list of labels."""
    return series.str.split(",").apply(lambda labels: [l.strip() for l in labels])

def create_balanced_sample(df, categories, n_per_category=N_PER_CATEGORY, min_per_combo=MIN_PER_COMBO, random_state=RANDOM_STATE):
    """
    Samples up to n_per_category rows per individual End product type label.
    A multi-label row (e.g. "Meat, Dairy") is eligible under each of its labels, and is kept
    only once in the combined sample if picked more than once. Then tops up the sample so at
    least min_per_combo rows of each distinct multi-label combination are present, to
    specifically test multi-label classification.
    """
    labels = parse_categories(df["End product type"])

    samples = []
    for cat in categories:
        mask = labels.apply(lambda xs: cat in xs)
        subset = df[mask]
        available = len(subset)
        if available == 0:
            print(f"  Warning: '{cat}' — no rows found, skipping.")
            continue
        n = min(n_per_category, available)
        if n < n_per_category:
            print(f"  Warning: '{cat}' — requested {n_per_category} but only {available} available, taking all.")
        samples.append(subset.sample(n=n, random_state=random_state))

    combined = pd.concat(samples) if samples else df.iloc[0:0]
    combined = combined[~combined.index.duplicated(keep="first")]

    multi_mask = labels.apply(lambda xs: len(xs) > 1)
    multi_labels = labels[multi_mask]
    combos = multi_labels.apply(lambda xs: ", ".join(sorted(xs)))

    for combo in combos.unique():
        combo_idx = combos[combos == combo].index
        group = df.loc[combo_idx]
        already = group.index.isin(combined.index).sum()
        need = min_per_combo - already
        if need <= 0:
            continue
        remaining_pool = group[~group.index.isin(combined.index)]
        take = min(need, len(remaining_pool))
        if take < need:
            print(f"  Warning: combo '{combo}' — only {already + len(remaining_pool)} rows available, wanted {min_per_combo}.")
        if take > 0:
            combined = pd.concat([combined, remaining_pool.sample(n=take, random_state=random_state)])

    combined = combined[~combined.index.duplicated(keep="first")]
    return combined.sample(frac=1, random_state=random_state).reset_index(drop=True)


In [5]:
endproduct_categories = breakdown.index.tolist()
test_data = create_balanced_sample(df, endproduct_categories)
print(f"test_data: {test_data.shape}")
# test_data[["Title", "Abstract", "End product type", "sub-end product"]]


test_data: (83, 81)


### 4. Save Subset to Excel
Allows manual check of files selected. Consider whether those in the test set are borderline cases or clear cut.


In [ ]:
################################################################################################
# PLEASE CHANGE FILENAME TO THE RANDOM SEED USED IN create_balanced_sample() FOR REPRODUCIBILITY
################################################################################################
def save_subset(df, filename, output_dir=RAW_SUBS_DIR):
    output_dir.mkdir(parents=True, exist_ok=True)
    path = output_dir / filename
    df.to_excel(path, index=False)
    print(f"Saved {len(df)} records to {path}")

save_subset(test_data, f"endproduct_test_data_rand{RANDOM_STATE}.xlsx")


Saved 83 records to 4_endproduct\raw_subsets\endproduct_test_data_rand3_2.xlsx


### 5. Load Prompt and Select Dataset


In [ ]:
PROMPT_VERSION = "v4"  # ← CHANGE THIS to switch prompt version
PROMPT_PATH = f"4_endproduct/{PROMPT_VERSION}/prompt_endproduct_grants_{PROMPT_VERSION}.md"

DATASET = test_data

# ONLY USED AS REQUIRED FOR RE-RUN SPECIFIC RECORDS
#ids_to_test = [10, 46]
#DATASET = DATASET[DATASET["id"].isin(ids_to_test)]
#DATASET = incorrect_endproduct_data
#DATASET = manual_test_data


In [46]:
def load_prompt(path=PROMPT_PATH):
    with open(path, "r", encoding="utf-8") as f:
        prompt_text = f.read()
    return prompt_text.strip()

system_prompt = load_prompt()
print(system_prompt)


You are an expert in alternative proteins and food technology.

Your task is to classify a grant on alternative proteins against a set of end product categories based on both its title and abstract.

Before assessing categories, identify every end product or application context that the grant's title and abstract explicitly support. A grant can genuinely target more than one end product type — in that case, flag every category that applies. Assign Agnostic only when the grant does not target any specific product type at all.

IMPORTANT: Base your classification ONLY on what the title and abstract explicitly state about the intended end product application. Do NOT infer an end product from external knowledge about how an ingredient, organism, or technology is commonly used. Do NOT infer a product category from the physical form of an ingredient (beads, microcapsules, spheres, gels, pastes) — a delivery system or encapsulated ingredient with a flavoured filling is not automatically confe

### 6. API Call with Prompt Caching


In [47]:
# API config

# Anthropic model options — pricing as of 2026-06-10.
# Verify at https://www.anthropic.com/pricing if costs may have changed.
# Model                  Input $/1M   Output $/1M   Context
# claude-haiku-4-5         $1.00         $5.00       200K
# claude-sonnet-4-6        $3.00        $15.00       1M
# claude-opus-4-8          $5.00        $25.00       1M
MODELS = {
    "haiku":  "claude-haiku-4-5",
    "sonnet": "claude-sonnet-4-6",
    "opus":   "claude-opus-4-8",
}
MODEL = MODELS["sonnet"]  # ← change this to switch model

MAX_TOKENS = 512         # max tokens in response; adjust based on expected reasoning length and cost tolerance
TEMPERATURE = 0.0        # 0.0 = deterministic; raise to ~0.3 to sample variance across REPETITIONS
CALL_DELAY = 1.0         # seconds between API calls
REQUEST_TIMEOUT = 120    # seconds before giving up on a single API call
MAX_RETRIES = 6          # retry attempts on rate-limit / transient errors
RETRY_BASE_SECONDS = 5.0  # exponential backoff base
RETRY_MAX_SECONDS = 90.0  # cap on backoff sleep

REPETITIONS = 1  # number of full runs; increase to measure output variance across runs

# ================================================================
# CHECKPOINT CONFIG
# Saves progress after each record; a run interrupted mid-way can
# be resumed without re-processing completed records. A dedicated
# subfolder keeps these runs from colliding with rescat checkpoints.
# Set RESUME_INCOMPLETE = False to always start from scratch.
# ================================================================
CHECKPOINT_DIR = Path("checkpoints/endproduct")
RESUME_INCOMPLETE = True


In [48]:
# ================================================================
# REASONING TOGGLE
# Keep True during testing — reasoning shows WHY the model decides
# as it does, which is essential for evaluating prompt quality.
# Set to False for production runs once the prompt is validated,
# to reduce token usage.
# ================================================================
INCLUDE_REASONING = True

ENDPRODUCT_CATS = [
    "Meat",
    "Fish and seafood",
    "Milk and milk proteins",
    "Yoghurt and fermented dairy",
    "Cheese",
    "Cream and ice cream",
    "Infant formula",
    "Dairy",
    "Agnostic",
    "Chocolate, desserts, and confectionery",
    "Eggs and egg proteins",
    "Spreads, sauces, and condiments",
]

# The 5 specific dairy subcategories the LLM flags individually — collapsed later into the
# ground truth's two-column layout ("End product type" = Dairy, "sub-end product" = the
# specific subcategory). The general "Dairy" flag is the ambiguous fallback and is NOT itself
# a sub-end product.
DAIRY_SUBCATS = ["Milk and milk proteins", "Yoghurt and fermented dairy", "Cheese", "Cream and ice cream", "Infant formula"]

import re

def make_schema(cats, include_reasoning):
    """
    Builds a schema with one boolean field per category (true/false, multi-label)
    instead of a single primary/secondary pick, since a grant can target more than
    one end product type at once. field_map translates the sanitised Python-safe
    field names (e.g. "Fish_and_seafood") back to the original category label
    (e.g. "Fish and seafood").
    """
    def field_name(cat):
        return re.sub(r"\W+", "_", cat).strip("_")

    field_map = {field_name(cat): cat for cat in cats}
    fields = {fname: (bool, ...) for fname in field_map}
    if include_reasoning:
        fields["reasoning"] = (str, ...)
    Model = create_model("ClassificationSchema", **fields)
    return Model, field_map

ClassificationSchema, CATEGORY_FIELD_MAP = make_schema(ENDPRODUCT_CATS, INCLUDE_REASONING)
print(f"Schema built: {ENDPRODUCT_CATS}")

client = anthropic.Anthropic(api_key=os.getenv("CLAUDE_API_KEY"))

def classify_publication(title, abstract, system_prompt):
    user_message = f"Title: {title}\n\nAbstract: {abstract}"
    response = client.messages.parse(
        model=MODEL,
        max_tokens=MAX_TOKENS,
        temperature=TEMPERATURE,
        timeout=REQUEST_TIMEOUT,
        system=[
            {
                "type": "text",
                "text": system_prompt,
                "cache_control": {"type": "ephemeral"}
            }
        ],
        messages=[
            {"role": "user", "content": user_message}
        ],
        output_format=ClassificationSchema,
    )
    return response.parsed_output


Schema built: ['Meat', 'Fish and seafood', 'Milk and milk proteins', 'Yoghurt and fermented dairy', 'Cheese', 'Cream and ice cream', 'Infant formula', 'Dairy', 'Agnostic', 'Chocolate, desserts, and confectionery', 'Eggs and egg proteins', 'Spreads, sauces, and condiments']


### 7. Error Handling with Retry


In [49]:
def is_retryable_error(exc: Exception) -> bool:
    markers = ["503", "UNAVAILABLE", "RESOURCE_EXHAUSTED", "429",
               "TIMEOUT", "TIMED OUT", "READTIMEOUT", "CONNECTTIMEOUT"]
    return any(m in str(exc).upper() for m in markers)

def retry_sleep_seconds(attempt: int) -> float:
    sleep = min(RETRY_MAX_SECONDS, RETRY_BASE_SECONDS * (2 ** attempt))
    jitter = random.uniform(0.0, min(3.0, sleep * 0.2))
    return sleep + jitter


In [50]:
def classify_with_error_handling(row, system_prompt):
    record_id = row["id"]
    last_error = None
    for attempt in range(MAX_RETRIES + 1):
        try:
            result = classify_publication(row["title"], row["abstract"], system_prompt)
            if result is None:
                print(f"  Parse failed for {record_id}: model returned no structured output")
                return {"id": record_id, "status": "parse_error", "error": "no structured output"}
            output = {f"{k}_LLM": v for k, v in result.model_dump().items()} # rename columns / keys to indicate LLM output
            output["id"] = record_id
            output["status"] = "ok"
            return output
        except anthropic.APIError as e:
            last_error = e
            if attempt >= MAX_RETRIES:
                break
            if is_retryable_error(e):
                sleep_s = retry_sleep_seconds(attempt)
                print(f"  Retryable error (attempt {attempt + 1}/{MAX_RETRIES}): {e}. Sleeping {sleep_s:.1f}s.")
                time.sleep(sleep_s)
            else:
                break
    print(f"  API error for {record_id}: {last_error}")
    return {"id": record_id, "status": "api_error", "error": str(last_error)}


### 8. Checkpoint Helpers


In [51]:
def get_checkpoint_path(run_idx: int) -> Path:
    CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
    return CHECKPOINT_DIR / f"run_{run_idx}_checkpoint.json"

def save_checkpoint(run_idx: int, completed_results: list) -> None:
    path = get_checkpoint_path(run_idx)
    payload = {
        "run_idx": run_idx,
        "completed_ids": [r["id"] for r in completed_results],
        "results": completed_results,
    }
    path.write_text(json.dumps(payload, ensure_ascii=False), encoding="utf-8")

def load_checkpoint(run_idx: int):
    path = get_checkpoint_path(run_idx)
    if not path.exists():
        return None
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except Exception:
        return None

def delete_checkpoint(run_idx: int) -> None:
    path = get_checkpoint_path(run_idx)
    if path.exists():
        path.unlink()


### 9. Run on Test Data


In [52]:
# Standardise DATASET column names once, separately from the LLM-calling loop below,
# so this (and the comparison cell) can be re-run for free without re-hitting the API
# — e.g. after a kernel restart, or when re-analysing results already in memory / a
# checkpoint file. Grants data has no reliable unique id column, so use row position.
DATASET = DATASET.reset_index(drop=True).rename(columns={"Title": "title", "Abstract": "abstract"})
DATASET["id"] = DATASET.index


In [53]:
all_results = []

for rep in range(REPETITIONS):
    run_idx = rep + 1
    print(f"\n{'='*50}\nRun {run_idx} / {REPETITIONS}\n{'='*50}")

    checkpoint = load_checkpoint(run_idx) if RESUME_INCOMPLETE else None
    if checkpoint:
        completed_results = checkpoint["results"]
        completed_ids = set(checkpoint["completed_ids"])
        print(f"  Resuming: {len(completed_ids)} records already processed.")
    else:
        completed_results, completed_ids = [], set()

    remaining = DATASET[~DATASET["id"].isin(completed_ids)]
    total = len(DATASET)

    for _, row in remaining.iterrows():
        n_done = len(completed_results)
        print(f"  [{n_done + 1}/{total}] {row['id']}")
        result = classify_with_error_handling(row, system_prompt)
        result["run"] = run_idx
        completed_results.append(result)
        save_checkpoint(run_idx, completed_results)
        if n_done + 1 < total:
            time.sleep(CALL_DELAY)

    delete_checkpoint(run_idx)
    all_results.extend(completed_results)



Run 1 / 1
  [1/5] 0
  [2/5] 1
  [3/5] 2
  [4/5] 3
  [5/5] 4


In [54]:
results_df = pd.DataFrame(all_results)
print(f"\nCompleted: {len(results_df)} records across {REPETITIONS} run(s)")
print(f"Successful: {(results_df['status'] == 'ok').sum()}")
print(f"Errors: {(results_df['status'] != 'ok').sum()}")
results_df



Completed: 5 records across 1 run(s)
Successful: 5
Errors: 0


,Meat_LLM,Fish_and_seafood_LLM,Milk_and_milk_proteins_LLM,Yoghurt_and_fermented_dairy_LLM,Cheese_LLM,Cream_and_ice_cream_LLM,Infant_formula_LLM,Dairy_LLM,Agnostic_LLM,Chocolate_desserts_and_confectionery_LLM,Eggs_and_egg_proteins_LLM,Spreads_sauces_and_condiments_LLM,reasoning_LLM,id,status,run
0,True,True,False,False,False,False,False,False,False,False,False,False,The grant explicitly targets whole cuts of fis...,0,ok,1
1,True,False,False,False,False,False,False,False,False,False,False,False,The grant explicitly targets tempeh-like mycel...,1,ok,1
2,False,False,False,False,False,False,False,False,True,False,False,False,The grant focuses on valorizing oat beverage p...,2,ok,1
3,False,False,False,False,False,False,False,False,True,False,False,False,The abstract describes a technology for produc...,3,ok,1
4,True,True,False,False,False,False,False,False,False,False,False,False,The grant explicitly targets cultivated octopu...,4,ok,1


In [55]:
# The v1-derived category taxonomy (kept as-is per review — see 4_endproduct/v2/NOTE.txt)
# uses different names than the ground truth's "End product type" vocabulary (Agnostic,
# Cocoa, Dairy, Eggs, Fats, Meat, Seafood, Snacks, Sugar). Two one-way mappings translate
# each side onto a shared comparison vocabulary so scoring compares like-for-like — neither
# mapping touches the raw/saved prediction or ground-truth columns, only the *_set columns
# used for scoring below.
NAME_MAP_TO_GROUND_TRUTH = {
    "Fish and seafood": "Seafood",
    "Eggs and egg proteins": "Eggs",
}

def to_ground_truth_name(cat):
    return NAME_MAP_TO_GROUND_TRUTH.get(cat, cat)

# Ground-truth-only buckets with no corresponding v2 category, mapped onto the closest v2
# category per review rather than left as permanent misses. Only Dairy has real
# sub-categories in this pipeline — Fats/Cocoa/Snacks/Sugar are collapsed straight onto a
# top-level category for comparison purposes, with no sub-end-product equivalent created.
GROUND_TRUTH_NAME_MAP_FOR_COMPARISON = {
    "Fats": "Meat",
    "Cocoa": "Chocolate, desserts, and confectionery",
    "Snacks": "Chocolate, desserts, and confectionery",
    "Sugar": "Chocolate, desserts, and confectionery",
}

def build_predictions(row):
    true_cats = [orig for field, orig in CATEGORY_FIELD_MAP.items() if row.get(f"{field}_LLM") == True]
    dairy_true = [c for c in true_cats if c in DAIRY_SUBCATS]
    non_dairy_true = [c for c in true_cats if c not in DAIRY_SUBCATS and c != "Dairy"]
    is_dairy = "Dairy" in true_cats or bool(dairy_true)

    end_product_type = [to_ground_truth_name(c) for c in non_dairy_true] + (["Dairy"] if is_dairy else [])

    # Joined with "; " rather than ", ": two of our own category names contain a comma
    # ("Chocolate, desserts, and confectionery", "Spreads, sauces, and condiments"), so a
    # comma-joined multi-label string would be ambiguous to split back apart. The ground
    # truth's own End product type/sub-end product columns stay comma-separated as-is
    # (their raw data format, and none of their category names contain a comma).
    return pd.Series({
        "raw_predicted_categories": "; ".join(true_cats),
        "predicted_end_product_type": "; ".join(end_product_type),
        "predicted_sub_end_product": "; ".join(dairy_true),
    })

results_df[["raw_predicted_categories", "predicted_end_product_type", "predicted_sub_end_product"]] = results_df.apply(build_predictions, axis=1)

result_cols = ["id", "run", "raw_predicted_categories", "predicted_end_product_type", "predicted_sub_end_product", "status"]
if INCLUDE_REASONING:
    result_cols.append("reasoning_LLM")

comparison = DATASET[["id", "title", "abstract", "End product type", "sub-end product"]].merge(
    results_df[result_cols], on="id", how="left"
)
comparison = comparison.rename(columns={"End product type": "end_product_type", "sub-end product": "sub_end_product"})

# Ground truth is comma-separated (its raw data format); our own predictions are
# semicolon-separated (see build_predictions above, re: category names containing commas).
# Compare both as sets so label order doesn't affect the match. name_map (when given)
# normalises labels onto the shared comparison vocabulary defined above.
def to_label_set(s, sep=",", name_map=None):
    if not isinstance(s, str) or not s.strip():
        return set()
    labels = {l.strip() for l in s.split(sep)}
    if name_map:
        labels = {name_map.get(l, l) for l in labels}
    return labels

comparison["end_product_type_set"] = comparison["end_product_type"].apply(lambda s: to_label_set(s, sep=",", name_map=GROUND_TRUTH_NAME_MAP_FOR_COMPARISON))
comparison["predicted_end_product_type_set"] = comparison["predicted_end_product_type"].apply(lambda s: to_label_set(s, sep=";"))
comparison["exact_match"] = comparison["end_product_type_set"] == comparison["predicted_end_product_type_set"]

comparison["sub_end_product_set"] = comparison["sub_end_product"].apply(lambda s: to_label_set(s, sep=","))
comparison["predicted_sub_end_product_set"] = comparison["predicted_sub_end_product"].apply(lambda s: to_label_set(s, sep=";"))
comparison["sub_exact_match"] = comparison["sub_end_product_set"] == comparison["predicted_sub_end_product_set"]

# Partial-credit metrics per row: even when the full label set doesn't match exactly,
# how much overlap is there between predicted and true labels?
def label_prf(truth_col, pred_col):
    def _prf(row):
        truth, pred = row[truth_col], row[pred_col]
        if not truth and not pred:
            return pd.Series({"precision": 1.0, "recall": 1.0, "jaccard": 1.0})
        tp = len(truth & pred)
        fp = len(pred - truth)
        fn = len(truth - pred)
        precision = tp / (tp + fp) if (tp + fp) else float("nan")
        recall = tp / (tp + fn) if (tp + fn) else float("nan")
        jaccard = tp / len(truth | pred) if (truth | pred) else float("nan")
        return pd.Series({"precision": precision, "recall": recall, "jaccard": jaccard})
    return _prf

comparison[["row_precision", "row_recall", "row_jaccard"]] = comparison.apply(label_prf("end_product_type_set", "predicted_end_product_type_set"), axis=1)
comparison[["sub_row_precision", "sub_row_recall", "sub_row_jaccard"]] = comparison.apply(label_prf("sub_end_product_set", "predicted_sub_end_product_set"), axis=1)

n = len(comparison)
print("== End product type ==")
print(f"Exact match accuracy:  {comparison['exact_match'].mean():.0%}  (n={n})")
print(f"Mean row precision:    {comparison['row_precision'].mean():.0%}")
print(f"Mean row recall:       {comparison['row_recall'].mean():.0%}")
print(f"Mean row Jaccard:      {comparison['row_jaccard'].mean():.0%}")

dairy_rows = comparison[comparison["end_product_type_set"].apply(lambda s: "Dairy" in s)]
print("\n== sub-end product (Dairy rows only) ==")
print(f"Exact match accuracy:  {dairy_rows['sub_exact_match'].mean():.0%}  (n={len(dairy_rows)})")
print(f"Mean row precision:    {dairy_rows['sub_row_precision'].mean():.0%}")
print(f"Mean row recall:       {dairy_rows['sub_row_recall'].mean():.0%}")
print(f"Mean row Jaccard:      {dairy_rows['sub_row_jaccard'].mean():.0%}")

# Spreads, sauces, and condiments has no ground-truth equivalent (no mapping target above)
# — it can never exact-match against ground truth, only ever show up as a false positive.
print("\nNote: 'Spreads, sauces, and condiments' has no ground-truth equivalent — it will never contribute to recall, only ever appear as an extra/false positive if predicted.")

# Per-category precision/recall across the multi-label predictions
def label_series(s, sep=","):
    return s.str.split(sep).explode().str.strip().dropna()

all_cats = sorted(set(label_series(comparison["end_product_type"], sep=",")) | set(label_series(comparison["predicted_end_product_type"], sep=";")))
cat_rows = []
for cat in all_cats:
    truth_has = comparison["end_product_type_set"].apply(lambda s: cat in s)
    pred_has  = comparison["predicted_end_product_type_set"].apply(lambda s: cat in s)
    tp = int((truth_has & pred_has).sum())
    fn = int((truth_has & ~pred_has).sum())
    fp = int((~truth_has & pred_has).sum())
    n_true = int(truth_has.sum())
    recall = tp / n_true if n_true else float("nan")
    precision = tp / (tp + fp) if (tp + fp) else float("nan")
    cat_rows.append({
        "end_product_type": cat, "n_true": n_true, "tp": tp, "fp": fp, "fn": fn,
        "recall": recall, "precision": precision,
    })
cat_stats = pd.DataFrame(cat_rows).set_index("end_product_type")
cat_stats["recall"] = cat_stats["recall"].map(lambda x: f"{x:.0%}" if pd.notna(x) else "-")
cat_stats["precision"] = cat_stats["precision"].map(lambda x: f"{x:.0%}" if pd.notna(x) else "-")
display(cat_stats)

# Detail table
display_cols = ["id", "title", "abstract", "end_product_type", "sub_end_product", "raw_predicted_categories", "predicted_end_product_type", "predicted_sub_end_product"]
if INCLUDE_REASONING:
    display_cols.append("reasoning_LLM")
display_cols += ["exact_match", "row_precision", "row_recall", "row_jaccard"]
comparison[display_cols]


== End product type ==
Exact match accuracy:  20%  (n=5)
Mean row precision:    40%
Mean row recall:       60%
Mean row Jaccard:      40%

== sub-end product (Dairy rows only) ==
Exact match accuracy:  0%  (n=2)
Mean row precision:    nan%
Mean row recall:       0%
Mean row Jaccard:      0%

Note: 'Spreads, sauces, and condiments' has no ground-truth equivalent — it will never contribute to recall, only ever appear as an extra/false positive if predicted.


,n_true,tp,fp,fn,recall,precision
end_product_type,,,,,,
Agnostic,0,0,2,0,-,0%
Dairy,2,0,0,2,0%,-
Meat,1,1,2,0,100%,33%
Seafood,2,2,0,0,100%,100%


,id,title,abstract,end_product_type,sub_end_product,raw_predicted_categories,predicted_end_product_type,predicted_sub_end_product,reasoning_LLM,exact_match,row_precision,row_recall,row_jaccard
0,0,Pioneering vegan whole cuts through\n myceliu...,Esencia Foods is the first company in Europe t...,Seafood,NaN,Meat; Fish and seafood,Meat; Seafood,,The grant explicitly targets whole cuts of fis...,False,0.5,1.0,0.5
1,1,The Root of Understanding Mycelia (RUMy),The project will investigate 32 different edib...,Meat,NaN,Meat,Meat,,The grant explicitly targets tempeh-like mycel...,True,1.0,1.0,1.0
2,2,Ingredient valorization of reOat – the fiber-r...,Purpose and goal: \nThe project aims to take p...,Dairy,Milk,Agnostic,Agnostic,,The grant focuses on valorizing oat beverage p...,False,0.0,0.0,0.0
3,3,Up-cycling of whey streams via microalgae\n i...,"Whey2blue, through the upcycling of dairy side...",Dairy,Milk,Agnostic,Agnostic,,The abstract describes a technology for produc...,False,0.0,0.0,0.0
4,4,Inktelligent Foods – enabling cultivated seafo...,Cultivated meat and seafood are a sustainable ...,Seafood,NaN,Meat; Fish and seafood,Meat; Seafood,,The grant explicitly targets cultivated octopu...,False,0.5,1.0,0.5


### 10. Save to Excel for Prompt Debugging
To assess how well the prompt does at getting the LLM to assign end product / sub-end product
labels, save the comparison data, then manually review what went wrong and adjust the prompt.
None of this will make it into the final workflow.

Order of working:
1. Create a new version folder in `4_endproduct` (e.g. `v3`).
2. Copy in the previous prompt. Label it with the new version number. Make updates as required based on step 6.
3. Edit Step 10 output directory (this step) and Step 5 prompt selection and input data.
4. Run the script from steps 5-10.
5. Manually review the results — both metrics and individual rows.
6. Write a text document (`NOTE.txt`) about the results and what changes you want to make to the prompt. Repeat from step 1.


In [ ]:
save_dir = Path(f"4_endproduct/{PROMPT_VERSION}")
save_dir.mkdir(parents=True, exist_ok=True)

summary_df = pd.DataFrame([
    {"metric": "end_product_type_exact_match_accuracy", "value": f"{comparison['exact_match'].mean():.0%}", "n": n},
    {"metric": "end_product_type_mean_precision", "value": f"{comparison['row_precision'].mean():.0%}", "n": n},
    {"metric": "end_product_type_mean_recall", "value": f"{comparison['row_recall'].mean():.0%}", "n": n},
    {"metric": "end_product_type_mean_jaccard", "value": f"{comparison['row_jaccard'].mean():.0%}", "n": n},
    {"metric": "sub_end_product_exact_match_accuracy (Dairy rows only)", "value": f"{dairy_rows['sub_exact_match'].mean():.0%}", "n": len(dairy_rows)},
    {"metric": "sub_end_product_mean_precision (Dairy rows only)", "value": f"{dairy_rows['sub_row_precision'].mean():.0%}", "n": len(dairy_rows)},
    {"metric": "sub_end_product_mean_recall (Dairy rows only)", "value": f"{dairy_rows['sub_row_recall'].mean():.0%}", "n": len(dairy_rows)},
    {"metric": "sub_end_product_mean_jaccard (Dairy rows only)", "value": f"{dairy_rows['sub_row_jaccard'].mean():.0%}", "n": len(dairy_rows)},
])

out_path = save_dir / f"endproduct_{PROMPT_VERSION}_{MODEL}_results.xlsx"
with pd.ExcelWriter(out_path) as writer:
    comparison[display_cols].to_excel(writer, sheet_name="results",     index=False)
    cat_stats.to_excel(             writer, sheet_name="by_category")
    summary_df.to_excel(            writer, sheet_name="summary",       index=False)

print(f"Saved to {out_path}")


Saved to 4_endproduct\v4\endproduct_v4_claude-sonnet-4-6_3results.xlsx


In [ ]:
# Records where the LLM's predicted End product type set did not exactly match ground
# truth — for re-run with a modified prompt
incorrect_ids = comparison.loc[~comparison["exact_match"], "id"]
incorrect_endproduct_data = DATASET[DATASET["id"].isin(incorrect_ids)].reset_index(drop=True)
incorrect_endproduct_data


In [20]:
# Manually select specific rows by id for quick re-testing (paste ids from the
# comparison/results tables above). Note: the Section 9 prep cell always resets
# "id" to a fresh 0..n-1 range when it runs, so once this subset goes through the
# script again its ids won't match the ones you selected here — use "title" or
# "end_product_type" to cross-reference back to the original run if needed.
manual_ids = [1,21,45,65,79]  # <- CHANGE THIS to the ids you want to re-test
manual_test_data = DATASET[DATASET["id"].isin(manual_ids)].reset_index(drop=True)
print(f"Selected {len(manual_test_data)} of {len(manual_ids)} requested ids")
manual_test_data


Selected 5 of 5 requested ids


,title,abstract,Original title,Database,Total amount,Gov contribution,Currency,Total amount (USD),Gov contribution (USD),Total amount (EUR),...,2027,2028,2029,2030,2031,2032,2033,2034,2035,id
0,Pioneering vegan whole cuts through\n myceliu...,Esencia Foods is the first company in Europe t...,NaN,airtable 2025,2020667,2020667,EUR,2144535.0,NaN,2020667.000,...,673555.6667,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
1,The Root of Understanding Mycelia (RUMy),The project will investigate 32 different edib...,NaN,groenprojektbank.dk/,12481420,12481420,DKK,NaN,NaN,1622584.600,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,21
2,Ingredient valorization of reOat – the fiber-r...,Purpose and goal: \nThe project aims to take p...,NaN,airtable,4541601,4541601,SEK,430358.0,NaN,413285.691,...,103321.4228,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,45
3,Up-cycling of whey streams via microalgae\n i...,"Whey2blue, through the upcycling of dairy side...",NaN,airtable 2025,518809,518809,CHF,582439.0,NaN,555125.630,...,138781.4075,138781.4075,NaN,NaN,NaN,NaN,NaN,NaN,NaN,65
4,Inktelligent Foods – enabling cultivated seafo...,Cultivated meat and seafood are a sustainable ...,NaN,airtable,300000,300000,EUR,327925.0,327925.0,300000.000,...,75000.0000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,79
